<a href="https://colab.research.google.com/github/nmb1106-wq/dsa405-project/blob/main/DSA405_002_FA26_P2_nmbrockm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DSA 405 – Project 2: Audit, Clean, and Document One Source

**Name:** Nicole Brockmole  
**Unity ID:** nmbrockm  
**Course:** DSA 405  
**Semester:** Fall 2026  

## Research Question

Do S&P 500 companies whose 10-K risk factor section (Item 1A) got a lot longer from one year to the next have worse stock returns over the following 12 months than companies whose risk section stayed about the same length?

## Dataset Used for P2

For Project 2, I am auditing and cleaning the S&P 500 constituent dataset. Each row represents a security currently included in the S&P 500. The dataset contains ticker symbols, company names, GICS sectors and sub-industries, headquarters locations, dates added to the index, SEC CIK identifiers, and founding information.

This dataset is important to my project because the CIK can be used to identify companies in SEC EDGAR, while the ticker symbol can later be used to connect companies to stock-price data.

In [91]:
import pandas as pd
import numpy as np

DATA_URL = "https://raw.githubusercontent.com/nmb1106-wq/dsa405-project/main/data/raw/constituents.csv"

df_raw = pd.read_csv(DATA_URL)
df = df_raw.copy()

df_raw.head()

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


## 1. Initial Data Audit

I first audited the raw dataset before making any changes. The audit examines the dataset's dimensions, data types, missing values, distinct values, numeric ranges, categorical values, duplicate records, identifiers, date formats, whitespace, sentinel values, and other possible data-quality problems.

In [92]:
print("Number of rows:", df_raw.shape[0])
print("Number of columns:", df_raw.shape[1])

print("\nColumn names:")
print(df_raw.columns.tolist())

print("\nLoaded data types:")
print(df_raw.dtypes)

Number of rows: 503
Number of columns: 8

Column names:
['Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry', 'Headquarters Location', 'Date added', 'CIK', 'Founded']

Loaded data types:
Symbol                   object
Security                 object
GICS Sector              object
GICS Sub-Industry        object
Headquarters Location    object
Date added               object
CIK                       int64
Founded                  object
dtype: object


In [93]:
desired_types = {
    "Symbol": "string",
    "Security": "string",
    "GICS Sector": "category",
    "GICS Sub-Industry": "category",
    "Headquarters Location": "string",
    "Date added": "datetime",
    "CIK": "string (10-character identifier)",
    "Founded": "string"
}

type_audit = pd.DataFrame({
    "Loaded dtype": df_raw.dtypes.astype(str),
    "Desired dtype": pd.Series(desired_types)
})

type_audit

,Loaded dtype,Desired dtype
Symbol,object,string
Security,object,string
GICS Sector,object,category
GICS Sub-Industry,object,category
Headquarters Location,object,string
Date added,object,datetime
CIK,int64,string (10-character identifier)
Founded,object,string


In [94]:
missing_audit = pd.DataFrame({
    "Missing Count": df_raw.isna().sum(),
    "Missing Rate (%)": (df_raw.isna().mean() * 100).round(2)
})

missing_audit

,Missing Count,Missing Rate (%)
Symbol,0,0.0
Security,0,0.0
GICS Sector,0,0.0
GICS Sub-Industry,0,0.0
Headquarters Location,0,0.0
Date added,0,0.0
CIK,0,0.0
Founded,0,0.0


In [95]:
distinct_audit = pd.DataFrame({
    "Distinct Values": df_raw.nunique(dropna=False),
    "Missing Count": df_raw.isna().sum(),
    "Missing Rate (%)": (df_raw.isna().mean() * 100).round(2)
})

distinct_audit

,Distinct Values,Missing Count,Missing Rate (%)
Symbol,503,0,0.0
Security,503,0,0.0
GICS Sector,11,0,0.0
GICS Sub-Industry,126,0,0.0
Headquarters Location,249,0,0.0
Date added,374,0,0.0
CIK,500,0,0.0
Founded,194,0,0.0


In [96]:
print("GICS Sector values:")
print(sorted(df_raw["GICS Sector"].unique()))

print("\nGICS Sector counts:")
print(df_raw["GICS Sector"].value_counts(dropna=False))

GICS Sector values:
['Communication Services', 'Consumer Discretionary', 'Consumer Staples', 'Energy', 'Financials', 'Health Care', 'Industrials', 'Information Technology', 'Materials', 'Real Estate', 'Utilities']

GICS Sector counts:
GICS Sector
Industrials               83
Financials                76
Information Technology    74
Health Care               60
Consumer Discretionary    47
Consumer Staples          33
Utilities                 31
Real Estate               30
Materials                 25
Communication Services    23
Energy                    21
Name: count, dtype: int64


In [97]:
print("CIK minimum:", df_raw["CIK"].min())
print("CIK maximum:", df_raw["CIK"].max())
print("CIK numeric range:", df_raw["CIK"].max() - df_raw["CIK"].min())

CIK minimum: 1800
CIK maximum: 2115436
CIK numeric range: 2113636


### CIK Interpretation

CIK loads as a numeric column, so its minimum, maximum, and numeric range are shown above as part of the audit. However, CIK is an SEC identifier rather than a quantitative measurement. Arithmetic on CIK values is not meaningful. For the cleaned dataset, CIK will therefore be stored as a 10-character string with leading zeros preserved.

In [98]:
print("Exact duplicate rows:", df_raw.duplicated().sum())
print("Duplicate ticker symbols:", df_raw["Symbol"].duplicated().sum())
print("Duplicate company names:", df_raw["Security"].duplicated().sum())
print("Duplicate CIK values:", df_raw["CIK"].duplicated().sum())

Exact duplicate rows: 0
Duplicate ticker symbols: 0
Duplicate company names: 0
Duplicate CIK values: 3


In [99]:
duplicate_ciks = (
    df_raw[
        df_raw.duplicated(subset="CIK", keep=False)
    ]
    .sort_values("CIK")
)

duplicate_ciks[["Symbol", "Security", "CIK"]]

,Symbol,Security,CIK
332,NWS,News Corp (Class B),1564708
331,NWSA,News Corp (Class A),1564708
19,GOOGL,Alphabet Inc. (Class A),1652044
20,GOOG,Alphabet Inc. (Class C),1652044
205,FOX,Fox Corporation (Class B),1754301
204,FOXA,Fox Corporation (Class A),1754301


### Duplicate-Key Assessment

There are no exact duplicate rows, duplicate ticker symbols, or duplicate security names. However, three CIK values occur more than once, affecting six rows. Inspection shows that these repeated CIKs correspond to companies with multiple listed share classes. Therefore, they are legitimate observations rather than duplicate records.

`Symbol` is the appropriate unique row-level identifier for this dataset. CIK identifies the SEC registrant, so the same CIK can legitimately appear for more than one listed security. No rows were removed because of repeated CIK values.

In [100]:
sentinel_values = [
    "NA", "N/A", "NULL", "null",
    "Unknown", "-", "?", "999", "9999"
]

sentinel_counts = {}

for col in df_raw.columns:
    count = (
        df_raw[col]
        .astype(str)
        .str.strip()
        .isin(sentinel_values)
        .sum()
    )
    sentinel_counts[col] = count

pd.Series(sentinel_counts, name="Sentinel values found")

,Sentinel values found
Symbol,0
Security,0
GICS Sector,0
GICS Sub-Industry,0
Headquarters Location,0
Date added,0
CIK,0
Founded,0


In [101]:
total_subtotal_mask = df_raw.astype(str).apply(
    lambda col: col.str.contains(
        r"\b(?:total|subtotal|grand total)\b",
        case=False,
        na=False
    )
).any(axis=1)

print("Total/subtotal rows found:", total_subtotal_mask.sum())

Total/subtotal rows found: 0


In [102]:
text_columns = df_raw.select_dtypes(include="object").columns

whitespace_counts = {}

for col in text_columns:
    count = (
        df_raw[col].astype(str)
        != df_raw[col].astype(str).str.strip()
    ).sum()

    whitespace_counts[col] = count

pd.Series(whitespace_counts, name="Values with leading/trailing whitespace")

,Values with leading/trailing whitespace
Symbol,0
Security,0
GICS Sector,0
GICS Sub-Industry,0
Headquarters Location,0
Date added,0
Founded,0


In [103]:
parsed_dates = pd.to_datetime(
    df_raw["Date added"],
    errors="coerce"
)

date_format_check = (
    df_raw["Date added"]
    .astype(str)
    .str.fullmatch(r"\d{4}-\d{2}-\d{2}")
)

print("Dates that failed to parse:", parsed_dates.isna().sum())
print("Dates using YYYY-MM-DD format:", date_format_check.sum())
print("Dates using another format:", (~date_format_check).sum())

Dates that failed to parse: 0
Dates using YYYY-MM-DD format: 503
Dates using another format: 0


In [104]:
encoding_issue_counts = {}

for col in df_raw.select_dtypes(include="object").columns:
    text = df_raw[col].astype(str)

    issue_count = (
        text.str.contains("\ufffd", regex=False)
        | text.str.contains(r"Ã|Â|â€", regex=True)
    ).sum()

    encoding_issue_counts[col] = issue_count

pd.Series(
    encoding_issue_counts,
    name="Possible encoding problems"
)

,Possible encoding problems
Symbol,0
Security,0
GICS Sector,0
GICS Sub-Industry,0
Headquarters Location,0
Date added,0
Founded,0


In [105]:
ciks_needing_padding = (
    df_raw["CIK"]
    .astype(str)
    .str.len()
    .lt(10)
    .sum()
)

print(
    "CIKs requiring leading-zero padding:",
    ciks_needing_padding
)

CIKs requiring leading-zero padding: 503


In [106]:
founded_numeric = pd.to_numeric(
    df_raw["Founded"],
    errors="coerce"
)

print(
    "Founded values that can be read as a single number:",
    founded_numeric.notna().sum()
)

print(
    "Founded values containing additional text/history:",
    founded_numeric.isna().sum()
)

print("\nExamples of non-numeric Founded values:")
print(
    df_raw.loc[
        founded_numeric.isna(),
        "Founded"
    ].drop_duplicates().head(10)
)

Founded values that can be read as a single number: 464
Founded values containing additional text/history: 39

Examples of non-numeric Founded values:
3                                   2013 (1888)
23                                  2019 (1860)
35                                  1982 (1919)
48                                  1983 (1885)
57                           1998 (1923 / 1874)
68                                  2007 (1784)
72                                  1989 (1887)
85     2020 (1915, United Technologies spinoff)
155                                 2019 (1897)
159                                 2017 (1802)
Name: Founded, dtype: object


### Defect Classes Checked

The audit found that the source is relatively clean.

- **Missing values:** 0 were found.
- **Sentinel values:** No common sentinel values such as N/A, NULL, Unknown, 999, or 9999 were found.
- **Total or subtotal rows:** 0 were found.
- **Exact duplicate rows:** 0 were found.
- **Duplicate ticker symbols:** 0 were found. All 503 symbols are unique.
- **Duplicate CIK values:** Three CIK values repeat across six rows because some companies have multiple share classes. These records were retained.
- **Leading zeros in identifiers:** All 503 CIK values require leading zeros to be represented in the standard 10-character format.
- **Leading or trailing whitespace:** None was found in the text columns.
- **Date formats:** All 503 dates use the YYYY-MM-DD format and successfully parse as dates.
- **Encoding problems:** No obvious replacement characters or common encoding problems were found.
- **Numbers stored as text:** `Founded` contains many year-like values but also includes historical descriptions and multiple-year entries, so it is intentionally retained as text.

The main cleaning needs are therefore data-type corrections, standardized column names, and proper formatting of the CIK identifier rather than removal of observations.

## 2. Data Dictionary

The following dictionary describes all eight variables used from this source. Variable names reflect the cleaned dataset.

In [107]:
data_dictionary = pd.DataFrame({
    "Variable name": [
        "symbol",
        "security",
        "gics_sector",
        "gics_sub_industry",
        "headquarters_location",
        "date_added",
        "cik",
        "founded"
    ],

    "Type": [
        "string",
        "string",
        "category",
        "category",
        "string",
        "date",
        "string",
        "string"
    ],

    "Units": [
        "None",
        "None",
        "None",
        "None",
        "None",
        "Calendar date",
        "None",
        "None"
    ],

    "Factor levels": [
        "N/A",
        "N/A",
        "11 observed GICS sectors",
        "126 observed GICS sub-industries",
        "N/A",
        "N/A",
        "N/A",
        "N/A"
    ],

    "Valid range / values": [
        "Non-empty ticker symbol",
        "Non-empty company/security name",
        "Documented GICS sector",
        "Documented GICS sub-industry",
        "Valid headquarters location",
        "Valid calendar date",
        "Exactly 10 numeric characters",
        "Source-provided founding year or description"
    ],

    "Missingness": [
        "0; missing would prevent ticker-based matching",
        "0; missing would remove the company label",
        "0; missing would mean sector is unknown",
        "0; missing would mean sub-industry is unknown",
        "0; missing would mean headquarters is unknown",
        "0; missing would mean index-addition date is unknown",
        "0; missing would prevent SEC CIK-based matching",
        "0; missing would mean founding information is unavailable"
    ],

    "Notes": [
        "Unique row-level identifier in this dataset.",
        "Company or listed security name.",
        "Broad GICS industry classification.",
        "More specific GICS industry classification.",
        "Location reported for company headquarters.",
        "Date the security was added to the S&P 500.",
        "SEC registrant identifier. It may repeat when one registrant has multiple share classes.",
        "Kept as text because some entries contain multiple years or historical descriptions."
    ]
})

data_dictionary

,Variable name,Type,Units,Factor levels,Valid range / values,Missingness,Notes
0,symbol,string,None,N/A,Non-empty ticker symbol,0; missing would prevent ticker-based matching,Unique row-level identifier in this dataset.
1,security,string,None,N/A,Non-empty company/security name,0; missing would remove the company label,Company or listed security name.
2,gics_sector,category,None,11 observed GICS sectors,Documented GICS sector,0; missing would mean sector is unknown,Broad GICS industry classification.
3,gics_sub_industry,category,None,126 observed GICS sub-industries,Documented GICS sub-industry,0; missing would mean sub-industry is unknown,More specific GICS industry classification.
4,headquarters_location,string,None,N/A,Valid headquarters location,0; missing would mean headquarters is unknown,Location reported for company headquarters.
5,date_added,date,Calendar date,N/A,Valid calendar date,0; missing would mean index-addition date is u...,Date the security was added to the S&P 500.
6,cik,string,None,N/A,Exactly 10 numeric characters,0; missing would prevent SEC CIK-based matching,SEC registrant identifier. It may repeat when ...
7,founded,string,None,N/A,Source-provided founding year or description,0; missing would mean founding information is ...,Kept as text because some entries contain mult...


### Data Dictionary Notes

No observed values clearly contradict the expected documentation for these variables. The main type issue is CIK: pandas initially loads it as an integer, but it is an identifier and should be represented as a 10-character string. `Founded` is intentionally treated as text because the source sometimes provides historical descriptions rather than a single numeric year.

## 3. Data Cleaning

Based on the audit, only proportionate cleaning is necessary. No observations need to be removed. The cleaning below standardizes column names, assigns appropriate data types, and restores the 10-character format of SEC CIK identifiers.

In [108]:
df = df_raw.copy()

df = df.rename(columns={
    "Symbol": "symbol",
    "Security": "security",
    "GICS Sector": "gics_sector",
    "GICS Sub-Industry": "gics_sub_industry",
    "Headquarters Location": "headquarters_location",
    "Date added": "date_added",
    "CIK": "cik",
    "Founded": "founded"
})

In [109]:
df["date_added"] = pd.to_datetime(
    df["date_added"],
    errors="raise"
)

df["cik"] = (
    df["cik"]
    .astype(str)
    .str.zfill(10)
)

df["gics_sector"] = (
    df["gics_sector"]
    .astype("category")
)

df["gics_sub_industry"] = (
    df["gics_sub_industry"]
    .astype("category")
)

text_cols = [
    "symbol",
    "security",
    "headquarters_location",
    "founded"
]

for col in text_cols:
    df[col] = df[col].astype("string")

In [110]:
print("Cleaned rows:", df.shape[0])
print("Cleaned columns:", df.shape[1])
print("Missing values:", df.isna().sum().sum())
print("Exact duplicates:", df.duplicated().sum())
print("Unique symbols:", df["symbol"].nunique())
print("Unique CIKs:", df["cik"].nunique())

print("\nCleaned data types:")
print(df.dtypes)

print("\nCIK length counts:")
print(df["cik"].str.len().value_counts())

Cleaned rows: 503
Cleaned columns: 8
Missing values: 0
Exact duplicates: 0
Unique symbols: 503
Unique CIKs: 500

Cleaned data types:
symbol                   string[python]
security                 string[python]
gics_sector                    category
gics_sub_industry              category
headquarters_location    string[python]
date_added               datetime64[ns]
cik                              object
founded                  string[python]
dtype: object

CIK length counts:
cik
10    503
Name: count, dtype: int64


In [111]:
assert len(df) == len(df_raw), "Unexpected row loss occurred."
assert df.shape[1] == df_raw.shape[1], "Unexpected column count change occurred."
assert df["symbol"].is_unique, "Symbol should uniquely identify each row."
assert df["cik"].str.fullmatch(r"\d{10}").all(), "Every CIK should contain exactly 10 digits."
assert df["date_added"].notna().all(), "All dates should remain valid after conversion."
assert df.isna().sum().sum() == 0, "Cleaning introduced unexpected missing values."

print("All cleaning validation checks passed.")

All cleaning validation checks passed.


### Tidy Data Assessment

The raw dataset was already in tidy form. Each row represents one S&P 500 security, each column represents one variable, and each cell contains one value. Therefore, no pivoting, melting, or other reshaping was necessary. My cleaning focused on data types, identifier formatting, and consistent column names rather than changing the structure of the dataset.

## 4. Cleaning Log

The cleaning log records both changes and important judgment calls. Because the source was already well structured, no rows were deleted. The raw file remains unchanged, and every transformation below can be reversed by returning to the raw source.

In [112]:
cleaning_log = pd.DataFrame([
    {
        "#": 1,
        "Column": "All 8 columns",
        "Change made": "Renamed columns to lowercase snake_case",
        "Rows/cells affected": "8 column names",
        "Why": "Consistent machine-friendly names are easier to reference in later code and joins.",
        "What is lost": "Original capitalization and spacing in column names; recoverable from raw file."
    },
    {
        "#": 2,
        "Column": "date_added",
        "Change made": "Converted text dates to datetime",
        "Rows/cells affected": "503 cells",
        "Why": "The values represent calendar dates and all 503 successfully passed the date-format audit.",
        "What is lost": "Original text representation; the underlying dates are unchanged."
    },
    {
        "#": 3,
        "Column": "cik",
        "Change made": "Converted integer identifier to string",
        "Rows/cells affected": "503 cells",
        "Why": "CIK is an identifier, not a quantity, so arithmetic treatment is inappropriate.",
        "What is lost": "Numeric dtype only; no identifying information is lost."
    },
    {
        "#": 4,
        "Column": "cik",
        "Change made": "Padded CIK values with leading zeros to 10 characters",
        "Rows/cells affected": "503 cells",
        "Why": "All 503 CIKs were shorter than 10 characters after loading and require leading zeros for standardized identifier formatting.",
        "What is lost": "Nothing; leading zeros restore identifier formatting."
    },
    {
        "#": 5,
        "Column": "gics_sector",
        "Change made": "Converted from object/text to category",
        "Rows/cells affected": "503 cells",
        "Why": "The column contains a fixed set of 11 repeated sector classifications.",
        "What is lost": "Nothing; observed sector labels are unchanged."
    },
    {
        "#": 6,
        "Column": "gics_sub_industry",
        "Change made": "Converted from object/text to category",
        "Rows/cells affected": "503 cells",
        "Why": "The column represents repeated GICS classification levels rather than unrestricted text.",
        "What is lost": "Nothing; observed labels are unchanged."
    },
    {
        "#": 7,
        "Column": "symbol, security, headquarters_location, founded",
        "Change made": "Converted text columns to pandas string dtype",
        "Rows/cells affected": "2,012 cells",
        "Why": "These four variables contain text and should have an explicit text data type.",
        "What is lost": "Nothing; text values are unchanged."
    },
    {
        "#": 8,
        "Column": "cik",
        "Change made": "Retained repeated CIK values",
        "Rows/cells affected": "6 rows reviewed; 0 removed",
        "Why": "The repeated CIKs represent legitimate multiple share classes rather than duplicate observations.",
        "What is lost": "Nothing; retaining them preserves legitimate securities."
    },
    {
        "#": 9,
        "Column": "founded",
        "Change made": "Retained as text instead of converting to numeric year",
        "Rows/cells affected": "503 values reviewed; 0 values rewritten",
        "Why": "Some entries contain multiple years or historical descriptions, so forcing a numeric conversion would discard information.",
        "What is lost": "Nothing."
    },
    {
        "#": 10,
        "Column": "All rows",
        "Change made": "Retained all observations after duplicate and quality review",
        "Rows/cells affected": "503 rows reviewed; 0 removed",
        "Why": "No exact duplicates, subtotal rows, or other invalid observations requiring deletion were found.",
        "What is lost": "Nothing; no rows were removed."
    }
])

cleaning_log

,#,Column,Change made,Rows/cells affected,Why,What is lost
0,1,All 8 columns,Renamed columns to lowercase snake_case,8 column names,Consistent machine-friendly names are easier t...,Original capitalization and spacing in column ...
1,2,date_added,Converted text dates to datetime,503 cells,The values represent calendar dates and all 50...,Original text representation; the underlying d...
2,3,cik,Converted integer identifier to string,503 cells,"CIK is an identifier, not a quantity, so arith...",Numeric dtype only; no identifying information...
3,4,cik,Padded CIK values with leading zeros to 10 cha...,503 cells,All 503 CIKs were shorter than 10 characters a...,Nothing; leading zeros restore identifier form...
4,5,gics_sector,Converted from object/text to category,503 cells,The column contains a fixed set of 11 repeated...,Nothing; observed sector labels are unchanged.
5,6,gics_sub_industry,Converted from object/text to category,503 cells,The column represents repeated GICS classifica...,Nothing; observed labels are unchanged.
6,7,"symbol, security, headquarters_location, founded",Converted text columns to pandas string dtype,"2,012 cells",These four variables contain text and should h...,Nothing; text values are unchanged.
7,8,cik,Retained repeated CIK values,6 rows reviewed; 0 removed,The repeated CIKs represent legitimate multipl...,Nothing; retaining them preserves legitimate s...
8,9,founded,Retained as text instead of converting to nume...,503 values reviewed; 0 values rewritten,Some entries contain multiple years or histori...,Nothing.
9,10,All rows,Retained all observations after duplicate and ...,503 rows reviewed; 0 removed,"No exact duplicates, subtotal rows, or other i...",Nothing; no rows were removed.


## 5. Row and Column Accounting

The cleaning process did not remove observations or variables. The changes were limited to names, data types, and identifier formatting.

In [113]:
accounting = pd.DataFrame({
    "Measure": [
        "Rows in raw file",
        "Rows removed as exact duplicates",
        "Rows removed as near-duplicates",
        "Rows removed for other reasons",
        "Rows in cleaned file",
        "Columns in raw file",
        "Columns dropped",
        "Columns added",
        "Columns in cleaned file"
    ],

    "Count": [
        len(df_raw),
        0,
        0,
        0,
        len(df),
        df_raw.shape[1],
        0,
        0,
        df.shape[1]
    ]
})

accounting

,Measure,Count
0,Rows in raw file,503
1,Rows removed as exact duplicates,0
2,Rows removed as near-duplicates,0
3,Rows removed for other reasons,0
4,Rows in cleaned file,503
5,Columns in raw file,8
6,Columns dropped,0
7,Columns added,0
8,Columns in cleaned file,8


## 6. Provenance Brief

This dataset is a snapshot of the companies and securities included in the S&P 500, based on the Wikipedia List of S&P 500 companies table. Wikipedia contributors maintain the table as a public reference for S&P 500 membership and basic company information. Each row represents one security and includes its ticker symbol, company name, industry classification, headquarters location, date added to the index, SEC CIK identifier, and founding information.

For my project, the dataset mainly acts as a bridge between other sources. The CIK can identify companies in SEC EDGAR, while the ticker symbol can connect them to stock-price data. It can also describe the industries represented in the index.

However, the dataset is a current snapshot and cannot by itself establish S&P 500 membership for every historical year in my study. Companies that previously left the index may be absent. It also does not contain 10-K risk-factor text or stock returns, so those measurements must come from separate sources.

## 7. Final Validation

In [114]:
print("FINAL DATASET CHECK")
print("-" * 40)

print("Rows:", len(df))
print("Columns:", df.shape[1])
print("Total missing values:", df.isna().sum().sum())
print("Exact duplicate rows:", df.duplicated().sum())
print("Unique symbols:", df["symbol"].nunique())
print("Unique CIKs:", df["cik"].nunique())
print("All symbols unique:", df["symbol"].is_unique)
print("All CIKs 10 digits:", df["cik"].str.fullmatch(r"\d{10}").all())
print("All dates valid:", df["date_added"].notna().all())

print("\nFinal data types:")
print(df.dtypes)

print("\nFirst five cleaned rows:")
display(df.head())

FINAL DATASET CHECK
----------------------------------------
Rows: 503
Columns: 8
Total missing values: 0
Exact duplicate rows: 0
Unique symbols: 503
Unique CIKs: 500
All symbols unique: True
All CIKs 10 digits: True
All dates valid: True

Final data types:
symbol                   string[python]
security                 string[python]
gics_sector                    category
gics_sub_industry              category
headquarters_location    string[python]
date_added               datetime64[ns]
cik                              object
founded                  string[python]
dtype: object

First five cleaned rows:


,symbol,security,gics_sector,gics_sub_industry,headquarters_location,date_added,cik,founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,0000066740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,0000091142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,0000001800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,0001551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,0001467373,1989


## 8. P2 Self-Check

### Diagnosis & Data Dictionary
I conducted a per-column audit of data types, missingness, distinct values, categorical levels, duplicates, identifiers, date formats, whitespace, sentinel values, total/subtotal rows, encoding issues, and numeric-looking text. All eight variables are documented in the data dictionary with their types, units, factor levels, valid values, missingness, and notes.

### Cleaning Execution
Cleaning was limited to necessary changes: standardized column names, appropriate data types, and standardized CIK formatting. No valid observations were removed. Judgment calls, including retaining repeated CIKs and keeping Founded as text, are documented.

### Cleaning Log
Each cleaning decision records the affected rows or cells, the reason for the decision, and what information was lost. The raw file makes each transformation reversible.

### Provenance
The Provenance Brief identifies the source, producer, purpose, contents, uses, and limitations of the dataset in fewer than 200 words.

### Tidy Structure and Reproducibility
The source was already tidy, so no structural reshaping was necessary. The raw dataset is kept separate from the cleaned working copy.

## Final Submission Checklist

- [x] Systematic audit output included
- [x] Loaded and desired data types documented
- [x] Missingness and distinct values audited
- [x] Categorical levels reviewed
- [x] Defect classes checked and documented
- [x] Every identified problem has a count
- [x] Data dictionary covers all eight variables
- [x] Cleaning decisions documented
- [x] Cleaning log includes counts, reasons, and information lost
- [x] Row and column accounting reconciles
- [x] Tidy-data assessment included
- [x] Provenance Brief is under 200 words
- [x] Raw `constituents.csv` stored unmodified in `data/raw/`
- [x] `data/raw/SOURCES.md` completed
- [x] `README.md` completed
- [x] Notebook changed to load raw data reproducibly from GitHub
- [x] Notebook restarted and run top-to-bottom without errors
- [x] Final notebook saved to GitHub
- [x] Self-scored official rubric attached
- [x] Bench Check scheduled/completed